In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install py_vncorenlp underthesea pyserini faiss-cpu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 7.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.2/416.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.8/197.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os

def install_java():
  !apt update -qq
  !apt-get install -y openjdk-21-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
  !java -version

install_java()

42 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!git clone https://github.com/tommachilez/virag4fc.git
%cd /content/virag4fc/

Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 668, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 668 (delta 223), reused 173 (delta 93), pack-reused 344 (from 1)
Receiving objects: 100% (668/668), 5.40 MiB | 12.67 MiB/s, done.
Resolving deltas: 100% (447/447), done.
/content/fact-checking-data-framework-vn


In [5]:
VNCORE_MODEL_PATH = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
stopwords_path = f"{dataset_path}/stopwords-vi.txt"

# BM25 Evaluation

In [6]:
mining_dir = f"{dataset_path}/vn_mining_test"
test_queries = f"{dataset_path}/test_queries.tsv"
test_qrels = f"{dataset_path}/test_qrels.tsv"
seg_queries = f"{mining_dir}/segmented_test_queries.tsv"

In [ ]:
# Step 1: Preprocessing (Uses PyVnCoreNLP)
!python -m src.scripts.hard_negative_mining.segment_queries \
    --input_tsv {test_queries} \
    --output_tsv {seg_queries} \
    --vncorenlp_path {VNCORE_MODEL_PATH} \
    --stopwords_path {stopwords_path}

>>> Initializing NLP Processor...
2025-12-27 07:26:16 INFO  WordSegmenter:24 - Loading Word Segmentation model
>>> Processing /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/test_queries.tsv...
7685it [00:13, 564.89it/s] 
>>> Done. 7685 queries segmented and saved to /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining_test/segmented_test_queries.tsv


In [7]:
# Step 2: Retrieving & Evaluating (Uses Pyserini)
# Note: This is a separate process execution, so a new JVM is started.
!python -m src.scripts.hard_negative_mining.bm25_retrieval_eval \
    --preprocessed_dir {mining_dir} \
    --queries_tsv {seg_queries} \
    --qrels_tsv {test_qrels} \
    --top_k 100

2026-01-14 00:58:32.076860: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-14 00:58:32.082195: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-14 00:58:32.096506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768352312.121987    1606 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768352312.129292    1606 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768352312.149797    1606 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
import os
import csv
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from pyserini.pyclass import autoclass

# Configuration (Same as your existing notebook)
index_dir = f"{mining_dir}/pyserini_index"
output_bm25_run = f"{dataset_path}/bm25_run_file.txt"
top_k = 1000  # Recommend fetching more docs (e.g. 1000) for better hybrid fusion

def load_queries(path):
    queries = {}
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        for row in reader:
            if len(row) >= 2:
                queries[row[0]] = row[1].strip()
    return queries

# 1. Initialize Searcher
print(">>> Initializing Searcher...")
searcher = LuceneSearcher(index_dir)
searcher.set_bm25(k1=1.2, b=0.75)
try:
    JWhitespaceAnalyzer = autoclass('org.apache.lucene.analysis.core.WhitespaceAnalyzer')
    searcher.set_analyzer(JWhitespaceAnalyzer())
except Exception as e:
    print(f"Warning: {e}")

# 2. Run Retrieval and Save
queries = load_queries(seg_queries)
print(f">>> Writing BM25 run file to {output_bm25_run}...")

with open(output_bm25_run, 'w', encoding='utf-8') as f_out:
    for qid, query_text in tqdm(queries.items()):
        try:
            hits = searcher.search(query_text, k=top_k)
        except:
            hits = []

        for i, hit in enumerate(hits):
            # Format: qid \t docid \t rank \t score
            f_out.write(f"{qid}\t{hit.docid}\t{i+1}\t{hit.score:.6f}\n")

print(">>> Done.")

>>> Initializing Searcher...
>>> Writing BM25 run file to /content/drive/MyDrive/KLTN_Project/Datasets/bm25_run_file.txt...


100%|██████████| 7685/7685 [10:16<00:00, 12.47it/s]

>>> Done.
